# AfriVoices — P0/P1 : dev-long, bug de padding, fenêtres v1 (texte) vs v2 (logits)

Ce notebook mesure, **avant toute resoumission**, les trois points identifiés dans la revue du dépôt :

1. **Dev-long** (§4) : reconstruit un dev qui INCLUT les clips > 20s (jamais mesurés jusqu'ici —
   toutes les évals filtraient `duree > 20`), plus un dev-court élargi (~100 clips/langue).
2. **Bug de padding** (§5) : les soumissions décodent des logits *paddés en batch* alors que
   toutes les mesures dev décodaient clip par clip. On compare : un-par-un / batch paddé
   (= soumission actuelle) / batch + troncature. Si ça diverge, une partie du +0.10
   dev↔leaderboard s'explique et se corrige gratuitement.
3. **Fenêtres v1 vs v2** (§6-§7) sur le dev-long :
   - *direct* = clip entier en une passe (comportement de la soumission v9-2 à 0.40283) ;
   - *v1* = fenêtres fixes 15s/2s + recollage TEXTE (ta soumission "fenêtres" en évaluation) ;
   - *v2* = coupes au silence + dernière fenêtre alignée fin + **recollage des LOGITS**
     (tranchage au milieu du chevauchement, posé sur la coupe-silence) + UN seul décodage LM par clip.
4. **Projection leaderboard** (§7) : combine WER court/long avec les proportions de clips
   longs du test, et estime le LB de chaque variante par **delta autour de l'ancre 0.40283**
   (plus fiable que l'offset +0.10, qui contient déjà la pénalité des longs).
5. **Grille (α, β) par langue + beam élargi kln/mas** (§8, optionnel) sur les logits en cache.

**Session GPU (T4 suffit).** Ordre : §1 (installe → REDÉMARRER → relance §1) → §2 → ... → §8.
Durées indicatives : §4 ~10-20 min (lecture Drive), §6 ~30-45 min (GPU), §8 ~45-90 min (CPU).


## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib, subprocess
need = [m for m in ["kenlm", "pyctcdecode", "jiwer"] if importlib.util.find_spec(m) is None]
if need:
    print("installation de", need, "...")
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "jiwer"], check=False)
    subprocess.run(["pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime (Exécution > Redémarrer la session), puis relance cette cellule.")
else:
    print("✓ kenlm + pyctcdecode + jiwer disponibles — continue en §2")

## 2. CONFIG — le seul bloc à modifier

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"      # modèle de référence (celui du 0.40283)
LM_SUBDIR  = "lm_v2"
ALPHA, BETA = 0.7, 0.5              # réglage global actuel (référence des comparaisons)
N_COURT = 100                       # clips <=20s par langue (padding test, WER court, grille)
N_LONG  = 40                        # clips >20s  par langue (validation des fenêtres)
WIN_SEC, OVER_SEC, SEUIL_SEC, SEARCH_SEC = 15.0, 2.0, 20.0, 3.0
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
assert os.path.isdir(MODEL), f"modèle absent: {MODEL}"
assert os.path.isdir(LM), f"LM absent: {LM}"
print(f"modèle: {MODEL_NAME} | LM: {LM_SUBDIR} | alpha={ALPHA} beta={BETA}")

## 3. Modèle + décodeurs + utilitaires (dont les 2 fonctions de logits)

`logits1` = une passe SANS padding (comme toutes tes mesures dev).
`logits_batch_double` = le batching EXACT de tes soumissions (tri par durée, padding),
qui retourne pour chaque item `(logits_complets, n_vrai)` : décoder `logits_complets`
reproduit la soumission actuelle ; décoder `logits_complets[:n_vrai]` = version tronquée.

In [ ]:
import torch, io, base64, glob, re, numpy as np, soundfile as sf, librosa, tempfile
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "" if device == "cuda" else "⚠️ GPU fortement recommandé")

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

raw = [t for t, _ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|", " "): labels.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}: n += 1; labels.append("\u2047" * n)
    else: labels.append(t)
assert labels.count("") == 1 and labels.count(" ") == 1

def clean_text(t):
    t = (t or "").lower().strip()
    t = re.sub(r"[^\w\s\u0129\u0169\u00e1\u00e0\u00e2\u00e4\u00e9\u00e8\u00ea\u00eb\u00ed\u00ec\u00ee\u00ef\u00f3\u00f2\u00f4\u00f6\u00fa\u00f9\u00fb\u00fc\']", "", t)
    return re.sub(r"\s+", " ", t)

def duree_bytes(b):
    try: i = sf.info(io.BytesIO(b)); return i.frames / i.samplerate
    except Exception:
        try: i = sf.info(io.BytesIO(base64.b64decode(b))); return i.frames / i.samplerate
        except Exception: return 999.0

def decode_bytes(b):
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def unigrams(lang, top=50000):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w, _ in c.most_common(top)]

UNI = {l: unigrams(l) for l in ["swa", "kik", "luo", "kln", "mas", "som"]}
decoders = {l: build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{l}.bin",
                                unigrams=UNI[l], alpha=ALPHA, beta=BETA)
            for l in UNI}
print("✓ 6 décodeurs prêts (alpha, beta) =", (ALPHA, BETA))

MAX_SEC_BATCH = 160.0

def logits1(arr):
    """Une passe, batch de 1, AUCUN padding — le régime de toutes tes mesures dev."""
    fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        return model(**{k: v.to(device) for k, v in fe.items()}).logits[0].cpu().numpy()

def logits_batch_double(arrs):
    """Batching EXACT des soumissions (tri par durée, padding=True).
    Retourne [(logits_complets, n_vrai)] dans l'ordre d'entrée :
      - logits_complets  : padding inclus  = comportement soumission actuelle
      - logits_complets[:n_vrai] : tronqués à la vraie longueur (correctif)."""
    order = sorted(range(len(arrs)), key=lambda k: len(arrs[k]))
    out = [None] * len(arrs); i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and (j - i + 1) * (len(arrs[order[j]]) / 16000.0) <= MAX_SEC_BATCH:
            j += 1
        idxs = order[i:j]
        feats = processor.feature_extractor([arrs[k] for k in idxs], sampling_rate=16000,
                                            return_tensors="pt", padding=True)
        am = feats.get("attention_mask")
        with torch.no_grad():
            lg = model(**{k: v.to(device) for k, v in feats.items()}).logits.cpu().numpy()
        T_out = lg.shape[1]
        if am is not None:
            fr = am.sum(-1).numpy().astype(float); fmax = float(am.shape[1])
        else:
            fr = np.array([len(arrs[k]) for k in idxs], dtype=float); fmax = fr.max()
        for bi, k in enumerate(idxs):
            n_true = max(1, int(round(T_out * fr[bi] / fmax)))
            out[k] = (lg[bi], min(n_true, T_out))
        i = j
    return out

## 4. Reconstruction du dev : COURT (≤20s) **et** LONG (>20s)

Comme l'audit, mais **sans jeter les clips longs**. Unscripted en premier (c'est là que
vivent les longs). Le swahili n'existe pas dans `anvke/` : on récupère le dev-court swa
depuis `prepared_v5/swa` (le split d'éval habituel, seed 42) ; pour le swa long, `prepared`
a été filtré ≤20s à la préparation, donc pas de longs — la projection §7 l'estimera par
le ratio médian des langues mesurées. Lecture Drive : ~10-20 min.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

ANVKE = f"{BASE}/anvke"
ALIAS = {"kik": ["kik", "kikuyu", "gikuyu"], "luo": ["luo", "dholuo"], "kln": ["kln", "kalenjin"],
         "mas": ["mas", "maasai"], "som": ["som", "somali"]}
lang_dirs = {}
for d in os.listdir(ANVKE):
    dl = d.lower()
    for lang, als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs:
            lang_dirs[lang] = os.path.join(ANVKE, d)
print("langues anvke:", list(lang_dirs))

dev_court, dev_long = {}, {}
for lang, root in lang_dirs.items():
    allp = sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
    dv = [p for p in allp if "/dev/" in p or os.path.basename(p).startswith("dev")]
    if not dv: dv = allp[:4]
    dv = [p for p in dv if "unscripted" in os.path.basename(p)] + \
         [p for p in dv if "unscripted" not in os.path.basename(p)]
    court, long_ = [], []
    for pq in dv:
        if len(court) >= N_COURT and len(long_) >= N_LONG: break
        try:
            df = pd.read_parquet(pq)
        except Exception:
            continue
        df = df.sample(frac=1, random_state=42)
        for _, r in df.iterrows():
            t = (r.get("transcription") or "").strip()
            a = r.get("audio")
            b = a.get("bytes") if isinstance(a, dict) else a
            if not t or b is None: continue
            dur = duree_bytes(b)
            if dur <= 20 and len(court) < N_COURT:
                court.append((b, clean_text(t)))
            elif 20 < dur <= 120 and len(long_) < N_LONG:
                long_.append((b, clean_text(t), dur))
            if len(court) >= N_COURT and len(long_) >= N_LONG: break
    dev_court[lang], dev_long[lang] = court, long_
    dl_ = [d for _, _, d in long_]
    extra = f" (moy {np.mean(dl_):.0f}s, max {max(dl_):.0f}s)" if dl_ else "  ⚠️ AUCUN clip long trouvé dans le dev"
    print(f"{lang}: {len(court)} courts | {len(long_)} longs{extra}")

# --- swahili : dev-court depuis le split d'éval habituel (prepared_v5, seed 42) ---
try:
    from datasets import load_from_disk
    swa_ds = load_from_disk(f"{BASE}/prepared_v5/swa")
    sp = swa_ds.train_test_split(test_size=60, seed=42)
    rows = []
    for ex in sp["test"]:
        a = ex.get("audio")
        b = a.get("bytes") if isinstance(a, dict) else a
        if b is None: continue
        lab = [t for t in ex.get("labels", []) if t != -100]
        ref = clean_text(processor.tokenizer.decode(lab, group_tokens=False)) if lab else \
              clean_text((ex.get("transcription") or ""))
        if ref: rows.append((b, ref))
    dev_court["swa"] = rows[:N_COURT]
    dev_long["swa"] = []
    print(f"swa: {len(dev_court['swa'])} courts (split d'éval habituel) | 0 longs (prepared filtré ≤20s)")
except Exception as ex:
    print("swa indisponible (", str(ex)[:90], ") — on continue sur 5 langues ; mas/kln, les plus touchées, sont couvertes")

n_long_tot = sum(len(v) for v in dev_long.values())
assert n_long_tot > 0, "aucun clip long trouvé : vérifie le contenu de anvke/*/dev/"
print(f"\nTOTAL dev-long: {n_long_tot} clips — c'est le jeu qui manquait à toutes les mesures.")

## 5. TEST P0 — le décodage des logits paddés (batch) diverge-t-il du un-par-un ?

Sur ~20 clips courts/langue : (a) un-par-un = régime de tes mesures dev ;
(b) batch paddé complet = régime de tes **soumissions** ; (c) batch + troncature.
Si (b) ≠ (a) et (c) ≈ (a) → tes soumissions perdent des points sur du padding jamais
entraîné, correctif gratuit (déjà intégré au notebook de soumission v2).

In [ ]:
import jiwer

sample = []   # (lang, ref, arr)
for lang, rows_ in dev_court.items():
    for b, ref in rows_[:20]:
        try: sample.append((lang, ref, decode_bytes(b)))
        except Exception: pass
print(f"{len(sample)} clips courts pour le test padding")

refs  = [r for _, r, _ in sample]
arrs  = [a for _, _, a in sample]
langs = [l for l, _, _ in sample]

p_solo = []
for (l, _, a) in tqdm(sample, desc="(a) un-par-un"):
    p_solo.append(decoders[l].decode(logits1(a)))

pairs = logits_batch_double(arrs)
p_pad  = [decoders[langs[i]].decode(pairs[i][0])                 for i in range(len(sample))]  # (b)
p_trim = [decoders[langs[i]].decode(pairs[i][0][:pairs[i][1]])   for i in range(len(sample))]  # (c)

diff_ab = [i for i in range(len(sample)) if p_solo[i] != p_pad[i]]
diff_ac = [i for i in range(len(sample)) if p_solo[i] != p_trim[i]]
w_a, w_b, w_c = jiwer.wer(refs, p_solo), jiwer.wer(refs, p_pad), jiwer.wer(refs, p_trim)

print(f"\nclips où (batch paddé) ≠ (un-par-un) : {len(diff_ab)}/{len(sample)} = {len(diff_ab)/len(sample):.0%}")
print(f"clips où (batch tronqué) ≠ (un-par-un) : {len(diff_ac)}/{len(sample)} = {len(diff_ac)/len(sample):.0%}")
print(f"WER  un-par-un : {w_a:.4f} | batch paddé : {w_b:.4f} | batch tronqué : {w_c:.4f}")

for i in diff_ab[:5]:
    print(f"\n[{langs[i]}] fin un-par-un : ...{' '.join(p_solo[i].split()[-10:])}")
    print(f"[{langs[i]}] fin paddé     : ...{' '.join(p_pad[i].split()[-10:])}")

print("\n--- VERDICT ---")
if len(diff_ab) == 0:
    print("✓ Aucune divergence : le padding est bénin sur ce modèle. Rien à corriger (garde la troncature par principe).")
elif w_c <= w_b - 0.002:
    print(f"⚠️ Bug CONFIRMÉ : le padding coûte ~{w_b - w_c:.4f} de WER sur ce sample.")
    print("   Toutes tes soumissions passées l'ont subi ; la troncature (notebook soumission v2) le corrige gratuitement.")
else:
    print("Divergences textuelles mais impact WER faible sur ce sample — garde quand même la troncature (elle ne coûte rien).")

## 6. Cache des logits (une passe GPU) : direct / fenêtres v1 / fenêtres v2

Pour chaque clip **long** : logits *direct* (clip entier), transcription *v1*
(fenêtres fixes 15s/2s, décodage par fenêtre **sans troncature** — fidèle à ta soumission
"fenêtres", padding compris), logits *v2* (coupes au silence, dernière fenêtre alignée
fin, recollage des logits, tranché au milieu du chevauchement). Pour chaque clip **court** :
logits un-par-un (WER de référence + grille §8). Les fenêtres v2 font ≤ `WIN + OVER/2` = 16s,
donc toujours dans le régime < 20s vu à l'entraînement. ~30-45 min sur T4.

In [ ]:
# --- v1 : copie EXACTE de ta soumission fenêtres ---
def fenetres_v1(arr, sr=16000):
    w = int(WIN_SEC * sr); o = int(OVER_SEC * sr); step = w - o
    segs = []; i = 0
    while i < len(arr):
        segs.append(arr[i:i + w])
        if i + w >= len(arr): break
        i += step
    return segs

def recolle(textes):
    if not textes: return ""
    out = textes[0].split()
    for t in textes[1:]:
        mots = t.split()
        if not mots: continue
        best = 0
        for k in range(1, min(6, len(out), len(mots)) + 1):
            if out[-k:] == mots[:k]: best = k
        out += mots[best:]
    return " ".join(out)

# --- v2 : coupes au silence + dernière fenêtre alignée fin + recollage des LOGITS ---
def _rms(x, fl=400, hop=160):
    """Énergie RMS par trame de 25 ms (hop 10 ms)."""
    if len(x) < fl:
        return np.array([float(np.sqrt(np.mean(x ** 2) + 1e-12))]), hop
    nfr = 1 + (len(x) - fl) // hop
    idx = np.arange(nfr)[:, None] * hop + np.arange(fl)[None, :]
    fr = x[idx]
    return np.sqrt((fr ** 2).mean(axis=1) + 1e-12), hop

def fenetres_v2(arr, sr=16000):
    """Retourne [(début, fin)] en échantillons. Coupe au minimum d'énergie dans
    [fin_cible - SEARCH_SEC, fin_cible] ; chevauchement OVER_SEC centré sur la coupe ;
    dernière fenêtre = [len - WIN, len] (jamais de fenêtre courte)."""
    L = len(arr); w = int(WIN_SEC * sr); o = int(OVER_SEC * sr); s_s = int(SEARCH_SEC * sr)
    if L <= w: return [(0, L)]
    wins = []; cur = 0
    while L - cur > w:
        target = cur + w
        a = max(target - s_s, cur + o)
        seg = arr[a:min(target, L)]
        rms, hop = _rms(seg)
        cut = a + int(np.argmin(rms)) * hop + 200          # centre approx. de la trame la plus silencieuse
        cut = min(max(cut, cur + o), L - 1)
        wins.append((cur, min(cut + o // 2, L)))
        cur = max(cut - o // 2, 0)                          # avance d'au moins OVER/2 -> pas de boucle infinie
    wins.append((max(0, L - w), L))
    if len(wins) >= 2 and wins[-1][0] <= wins[-2][0]:       # la dernière englobe l'avant-dernière
        wins[-2] = (wins[-2][0], L); wins.pop()
    return wins

def stitch(wins, lgs):
    """Assemble les logits des fenêtres en une seule ligne de temps.
    Chaque chevauchement est tranché en son MILIEU — qui tombe sur la coupe-silence,
    donc sur des frames quasi vides : chaque frame provient d'UNE seule fenêtre, avec
    >= OVER/2 de contexte de part et d'autre. Les indices sont relatifs à chaque fenêtre
    (spf propre), donc AUCUNE dérive d'alignement, quelle que soit la durée du clip."""
    if len(wins) == 1:
        return lgs[0].astype(np.float32)
    parts = []
    for i, ((s, e), lg) in enumerate(zip(wins, lgs)):
        nfr = lg.shape[0]
        if nfr == 0: continue
        spf = (e - s) / nfr
        lo = s if i == 0 else (wins[i - 1][1] + s) // 2
        hi = e if i == len(wins) - 1 else (e + wins[i + 1][0]) // 2
        a = max(0, int(round((lo - s) / spf)))
        b = min(nfr, int(round((hi - s) / spf)))
        if b > a:
            parts.append(lg[a:b].astype(np.float32))
    return np.concatenate(parts, axis=0)

# ---------------- passe GPU ----------------
court_logits, cache = {}, {}
for lang, rows_ in dev_court.items():
    lst = []
    for b, ref in tqdm(rows_, desc=f"court {lang}", leave=False):
        try: arr = decode_bytes(b)
        except Exception: continue
        lst.append((logits1(arr).astype(np.float16), ref))
    court_logits[lang] = lst
    print(f"court {lang}: {len(lst)} logits en cache")

for lang, rows_ in dev_long.items():
    if not rows_: continue
    entries = []
    for b, ref, dur in tqdm(rows_, desc=f"long  {lang}", leave=False):
        try: arr = decode_bytes(b)
        except Exception: continue
        e = {"ref": ref, "dur": dur}
        try:
            e["direct"] = logits1(arr).astype(np.float16)
        except RuntimeError:
            torch.cuda.empty_cache(); e["direct"] = None       # OOM sur un très long clip
        segs1 = fenetres_v1(arr)                               # v1 fidèle (padding compris)
        if len(segs1) == 1 and e["direct"] is not None:
            e["txt_v1"] = decoders[lang].decode(e["direct"].astype(np.float32))
        else:
            pr1 = logits_batch_double(segs1)
            e["txt_v1"] = recolle([decoders[lang].decode(lg) for lg, _ in pr1])
        wins = fenetres_v2(arr)                                # v2 : logits tronqués + stitch
        pr2 = logits_batch_double([arr[ws:we] for ws, we in wins])
        e["v2"] = stitch(wins, [lg[:nv] for lg, nv in pr2]).astype(np.float16)
        e["n_wins_v2"] = len(wins)
        entries.append(e)
    cache[lang] = entries
    torch.cuda.empty_cache()
    print(f"long  {lang}: {len(entries)} clips en cache "
          f"({np.mean([e['n_wins_v2'] for e in entries]):.1f} fenêtres v2/clip en moy.)")

## 7. Comparaison sur le dev-long + projection leaderboard

WER par langue : *direct* vs *v1 (texte)* vs *v2 (logits)*. Puis projection :
`WER_projeté(langue) = (1-p_long)·WER_court + p_long·WER_long(variante)`, avec les
proportions de longs mesurées sur le TEST (rapport §9). L'estimation LB de chaque
variante = **0.40283 + (macro_projeté(variante) − macro_projeté(direct))** : on raisonne
en delta autour de l'ancre connue, ce qui neutralise l'offset dev↔LB.

In [ ]:
import jiwer

P_LONG = {"swa": 0.415, "mas": 0.361, "kln": 0.317, "som": 0.229, "luo": 0.196, "kik": 0.178}  # mesuré sur le TEST

# WER court de référence (réglage actuel), sur le dev-court élargi
wer_court = {}
for lang, its in court_logits.items():
    if not its: continue
    refs = [r for _, r in its]
    preds = [decoders[lang].decode(lg.astype(np.float32)) for lg, _ in its]
    wer_court[lang] = jiwer.wer(refs, preds)

res = {}
print(f"{'lang':5} {'n':>4} {'dur.moy':>8} | {'direct':>7} {'v1-texte':>9} {'v2-logits':>10}")
for lang, entries in cache.items():
    ok = [e for e in entries if e["direct"] is not None]
    refs_all = [e["ref"] for e in entries]
    d = jiwer.wer([e["ref"] for e in ok],
                  [decoders[lang].decode(e["direct"].astype(np.float32)) for e in ok]) if ok else float("nan")
    v1 = jiwer.wer(refs_all, [e["txt_v1"] for e in entries])
    v2 = jiwer.wer(refs_all, [decoders[lang].decode(e["v2"].astype(np.float32)) for e in entries])
    res[lang] = {"direct": d, "v1": v1, "v2": v2}
    if len(ok) < len(entries):
        print(f"  ({lang}: {len(entries)-len(ok)} clips 'direct' sautés pour OOM — WER direct sur le reste)")
    print(f"{lang:5} {len(entries):>4} {np.mean([e['dur'] for e in entries]):>7.0f}s | {d:>7.3f} {v1:>9.3f} {v2:>10.3f}")

variants = ["direct", "v1", "v2"]
ratio_med = {v: float(np.median([res[l][v] / wer_court[l] for l in res if l in wer_court and wer_court[l] > 0]))
             for v in variants}

print(f"\n{'lang':5} {'WERcourt':>9} {'p_long':>7} | " + " ".join(f"{'proj_'+v:>11}" for v in variants))
macro = {v: 0.0 for v in variants}; nlang = 0
for lang in ["swa", "som", "kik", "luo", "kln", "mas"]:
    wc = wer_court.get(lang)
    if wc is None: continue
    p = P_LONG[lang]; nlang += 1
    projs = {}
    for v in variants:
        wl = res[lang][v] if lang in res and not np.isnan(res[lang][v]) else wc * ratio_med[v]
        projs[v] = (1 - p) * wc + p * wl
        macro[v] += projs[v]
    star = "  (long estimé via ratio médian)" if lang not in res else ""
    print(f"{lang:5} {wc:>9.3f} {p:>6.0%} | " + " ".join(f"{projs[v]:>11.3f}" for v in variants) + star)
for v in variants: macro[v] /= max(nlang, 1)

LB_ANCRE = 0.40283
print(f"\nMACRO projeté ({nlang} langues) : " + " | ".join(f"{v} {macro[v]:.4f}" for v in variants))
print(f"Estimation leaderboard (delta vs direct, ancre {LB_ANCRE}) :")
for v in ["v1", "v2"]:
    print(f"  fenêtres {v}: ~{LB_ANCRE + (macro[v] - macro['direct']):.4f}  (delta {macro[v] - macro['direct']:+.4f})")
print("\nDécision : soumets la variante au meilleur macro projeté SI son delta < -0.005 ;")
print("entre -0.005 et 0, c'est du bruit à ces effectifs — augmente N_LONG avant de trancher.")

## 8. (Optionnel) Grille (α, β) PAR LANGUE + beam élargi kln/mas — sur les logits en cache

Ton notebook hybride avait prouvé que les optima diffèrent par langue (0.5/1.0 vs 0.7/0.5) ;
la lignée v9 est revenue à un réglage global jamais re-optimisé. Grille sur le mélange
court + long-v2 (représentatif de la soumission). CPU pur, ~45-90 min ; réduis
`N_GRID_PAR_LANGUE` ou la grille si pressé. **Règle anti-bruit** : n'adopte un réglage
par langue que si son gain vs (0.7, 0.5) dépasse ~0.015 à ces effectifs.

In [ ]:
import jiwer, random

A_GRID = [0.3, 0.5, 0.7, 0.9, 1.1]
B_GRID = [0.0, 0.5, 1.0, 2.0]
N_GRID_PAR_LANGUE = 80

grid_items = {}
for lang in set(list(court_logits) + list(cache)):
    its = [(lg, ref) for lg, ref in court_logits.get(lang, [])]
    its += [(e["v2"], e["ref"]) for e in cache.get(lang, [])]
    random.Random(42).shuffle(its)
    grid_items[lang] = its[:N_GRID_PAR_LANGUE]

best_ab, base_ab = {}, {}
for lang, its in grid_items.items():
    if not its: continue
    refs = [r for _, r in its]
    scores = {}
    for a in A_GRID:
        row = []
        for b in B_GRID:
            dec = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                   unigrams=UNI[lang], alpha=a, beta=b)
            scores[(a, b)] = jiwer.wer(refs, [dec.decode(lg.astype(np.float32)) for lg, _ in its])
            row.append(f"b={b}:{scores[(a,b)]:.3f}")
        print(f"{lang} a={a}: " + "  ".join(row))
    best_ab[lang] = min(scores, key=scores.get)
    base_ab[lang] = scores.get((0.7, 0.5), float("nan"))
    g = base_ab[lang] - scores[best_ab[lang]]
    verdict = "→ ADOPTER" if g > 0.015 else "→ bruit probable, garder (0.7,0.5)"
    print(f"{lang}: meilleur {best_ab[lang]} = {scores[best_ab[lang]]:.4f} | actuel (0.7,0.5) = {base_ab[lang]:.4f} | gain {g:+.4f} {verdict}\n")

print("À reporter dans le notebook de soumission v2 (uniquement les langues 'ADOPTER') :")
print("AB =", {l: best_ab[l] for l in best_ab})

# --- beam élargi sur les 2 langues faibles (marge RTF énorme : 0.28 vs 2.0) ---
print("\n--- beam élargi kln/mas ---")
for lang in ["kln", "mas"]:
    its = grid_items.get(lang)
    if not its: continue
    a, b = best_ab.get(lang, (ALPHA, BETA))
    dec = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin", unigrams=UNI[lang], alpha=a, beta=b)
    refs = [r for _, r in its]
    w_def = jiwer.wer(refs, [dec.decode(lg.astype(np.float32)) for lg, _ in its])
    w_big = jiwer.wer(refs, [dec.decode(lg.astype(np.float32), beam_width=256, token_min_logp=-7.0) for lg, _ in its])
    print(f"{lang} (a={a}, b={b}) : beam défaut {w_def:.4f} | beam_width=256, token_min_logp=-7 : {w_big:.4f} "
          f"({'→ à garder' if w_def - w_big > 0.01 else '→ pas de gain net'})")

## 9. Marche à suivre selon les chiffres

1. **§5 (padding)** : si le bug est confirmé, la troncature est déjà intégrée au notebook
   `afrivoices_soumission_fenetres_v2.ipynb` — rien d'autre à faire.
2. **§7 (fenêtres)** : soumets la variante gagnante avec le notebook v2 ; compare au
   leaderboard à **0.40283** (direct) et à ta soumission fenêtres v1. Le delta projeté
   de §7 est ta prédiction — note l'écart réel, il recalibrera les estimations suivantes.
3. **§8 (α, β par langue)** : reporte le dict `AB` (langues "ADOPTER" seulement) dans la
   §2 du notebook de soumission v2. Si kln/mas gagnent avec le beam élargi, la §3 du
   notebook v2 a un commutateur `BEAM_LANG` prêt.
4. Toujours ta règle : **décider avec le LM, sur le dev, chiffres en main** — et un seul
   changement par soumission si tu veux pouvoir attribuer les gains.